In [ ]:
"""
02_psf_plot_and_stats.py

PSF plotting (Bead-only vs. treatments) with outlier removal + stats figure.

What it does:
1) Reads per-condition XLSX files (sheet 0 = 20 ms, sheet 1 = 50 ms)
2) Removes outliers using IQR rule (per sample, per sheet; flags if X OR Y is outlier)
3) Saves cleaned data + outliers to disk (CSV)
4) Plots 20 ms and 50 ms as two separate high-resolution figures:
   - Left: FWHM_x vs FWHM_y scatter (plots max N_PLOT points per sample)
   - Right: boxplots for FWHM_x and FWHM_y (uses all cleaned points)
5) Creates an additional stats figure per exposure:
   - Boxplots of Combined FWHM = (FWHM_x + FWHM_y)/2
   - Pairwise Mann–Whitney U (two-sided) vs Bead + Holm correction
   - Asterisks shown above treatments (ns hidden)

Inputs expected:
Each XLSX file must contain columns:
- fwhm_x_nm_corr
- fwhm_y_nm_corr

Outputs:
- <BASE_DIR>/psf_outliers/*.csv
- <BASE_DIR>/psf_plots/*.png, *.pdf
- <BASE_DIR>/psf_clean_summary.csv
"""

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# =============================================================================
# USER SETTINGS (edit these)
# =============================================================================

# Folder containing the per-condition Excel files
BASE_DIR = Path("data/example_psf_tables")  # e.g., Path("data/example_psf_tables") or Path("/path/to/tables")

FILES = {
    "Bead": "Bead.xlsx",
    "Bead+CHCA": "Bead+CHCA.xlsx",
    "Bead+NH": "Bead+NH.xlsx",
    # add more conditions here if needed
}

# Sheets: 0 -> 20 ms, 1 -> 50 ms
SHEETS = {"20 ms": 0, "50 ms": 1}

COL_X = "fwhm_x_nm_corr"
COL_Y = "fwhm_y_nm_corr"

# Outlier rule
IQR_K = 1.5

# Scatter plotting
N_PLOT = 100
RANDOM_SEED = 7

# Output folders (created inside BASE_DIR)
OUTLIER_DIRNAME = "psf_outliers"
PLOT_DIRNAME = "psf_plots"
DPI = 600

# Colors (dark red theme used in your paper figures)
RED_LIGHT = "#ff2743"  # FWHM_x
RED_DARK = "#900c1d"   # FWHM_y

# =============================================================================


def ensure_dirs(base_dir: Path) -> tuple[Path, Path]:
    outlier_dir = base_dir / OUTLIER_DIRNAME
    plot_dir = base_dir / PLOT_DIRNAME
    outlier_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)
    return outlier_dir, plot_dir


def read_sheet(filepath: Path, sheet_idx: int) -> pd.DataFrame:
    df = pd.read_excel(filepath, sheet_name=sheet_idx, engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]
    if COL_X not in df.columns or COL_Y not in df.columns:
        raise ValueError(
            f"Missing required columns in {filepath.name} (sheet {sheet_idx}). "
            f"Expected: {COL_X}, {COL_Y}. Found: {list(df.columns)}"
        )

    df = df[[COL_X, COL_Y]].copy()
    df[COL_X] = pd.to_numeric(df[COL_X], errors="coerce")
    df[COL_Y] = pd.to_numeric(df[COL_Y], errors="coerce")
    df = df.dropna(subset=[COL_X, COL_Y]).reset_index(drop=True)
    return df


def iqr_bounds(series: pd.Series, k: float = 1.5) -> tuple[float, float]:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - k * iqr
    hi = q3 + k * iqr
    return float(lo), float(hi)


def split_outliers_iqr_2d(df: pd.DataFrame, k: float = 1.5) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Flag a row as outlier if X OR Y is outside its IQR bounds (per sample, per exposure)."""
    x_lo, x_hi = iqr_bounds(df[COL_X], k=k)
    y_lo, y_hi = iqr_bounds(df[COL_Y], k=k)

    mask_out = (
        (df[COL_X] < x_lo) | (df[COL_X] > x_hi) |
        (df[COL_Y] < y_lo) | (df[COL_Y] > y_hi)
    )

    outliers = df.loc[mask_out].copy()
    cleaned = df.loc[~mask_out].copy()

    outliers["outlier_reason"] = ""
    outliers.loc[(outliers[COL_X] < x_lo) | (outliers[COL_X] > x_hi), "outlier_reason"] += "X;"
    outliers.loc[(outliers[COL_Y] < y_lo) | (outliers[COL_Y] > y_hi), "outlier_reason"] += "Y;"
    outliers["x_lo"], outliers["x_hi"] = x_lo, x_hi
    outliers["y_lo"], outliers["y_hi"] = y_lo, y_hi

    return cleaned.reset_index(drop=True), outliers.reset_index(drop=True)


def sample_for_scatter(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed).copy()


def holm_correction(pvals: list[float]) -> list[float]:
    """Holm-Bonferroni correction (step-down). Returns adjusted p-values in original order."""
    m = len(pvals)
    order = np.argsort(pvals)
    p_sorted = np.array(pvals)[order]

    adj_sorted = np.empty(m, dtype=float)
    for i, p in enumerate(p_sorted):
        adj_sorted[i] = min(1.0, (m - i) * p)

    # enforce monotonicity
    for i in range(1, m):
        adj_sorted[i] = max(adj_sorted[i], adj_sorted[i - 1])

    adj = np.empty(m, dtype=float)
    adj[order] = adj_sorted
    return adj.tolist()


def p_to_stars(p: float) -> str:
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def stats_vs_bead(data_by_sample: dict[str, pd.DataFrame], metric_col: str) -> pd.DataFrame:
    """Each treatment vs Bead: Mann–Whitney U (two-sided) + Holm correction."""
    if "Bead" not in data_by_sample:
        raise ValueError("stats_vs_bead: 'Bead' sample not found.")

    bead_vals = data_by_sample["Bead"][metric_col].values
    comps, raw_pvals = [], []

    for sname, df in data_by_sample.items():
        if sname == "Bead":
            continue
        vals = df[metric_col].values
        _, p = mannwhitneyu(bead_vals, vals, alternative="two-sided")
        comps.append(sname)
        raw_pvals.append(float(p))

    adj_pvals = holm_correction(raw_pvals)

    out = []
    for sname, p_raw, p_adj in zip(comps, raw_pvals, adj_pvals):
        out.append(
            {
                "comparison": f"{sname} vs Bead",
                "p_raw": p_raw,
                "p_adj_holm": p_adj,
                "stars": p_to_stars(p_adj),
            }
        )
    return pd.DataFrame(out)


def make_exposure_figure(exposure_label: str, data_by_sample: dict[str, pd.DataFrame], plot_dir: Path) -> None:
    """
    Creates a figure per exposure:
      Left: scatter (max N_PLOT points per sample)
      Right: boxplots for X and Y (all cleaned)
    """
    np.random.seed(RANDOM_SEED + (0 if exposure_label == "20 ms" else 100))

    samples = list(data_by_sample.keys())

    # Scatter: sample down for visibility
    scatter_samples = {
        s: sample_for_scatter(data_by_sample[s], n=N_PLOT, seed=RANDOM_SEED) for s in samples
    }

    x_arrays = [data_by_sample[s][COL_X].values for s in samples]
    y_arrays = [data_by_sample[s][COL_Y].values for s in samples]

    plt.rcParams.update(
        {
            "font.family": "Arial",
            "font.size": 12,
            "axes.titlesize": 22,
            "axes.labelsize": 18,
            "xtick.labelsize": 14,
            "ytick.labelsize": 14,
            "legend.fontsize": 10,
        }
    )

    fig = plt.figure(figsize=(12.5, 5.8))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.25, 1.0])

    # ---------------- Scatter ----------------
    ax0 = fig.add_subplot(gs[0, 0])
    for s in samples:
        dfp = scatter_samples[s]
        ax0.scatter(dfp[COL_X], dfp[COL_Y], s=18, alpha=0.75, label=s)

    ax0.set_xlabel("FWHM$_x$ (nm)")
    ax0.set_ylabel("FWHM$_y$ (nm)")
    ax0.set_title(f"PSF Scatter • {exposure_label}")
    ax0.grid(True, alpha=0.25)

    # Legend outside scatter
    ax0.legend(frameon=True, loc="upper left", bbox_to_anchor=(1.02, 1.0))

    # ---------------- Boxplots ----------------
    ax1 = fig.add_subplot(gs[0, 1])

    positions_x = np.arange(len(samples)) * 2.0
    positions_y = positions_x + 0.7

    bp1 = ax1.boxplot(
        x_arrays,
        positions=positions_x,
        widths=0.55,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(linewidth=1.6),
    )
    bp2 = ax1.boxplot(
        y_arrays,
        positions=positions_y,
        widths=0.55,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(linewidth=1.6),
    )

    for b in bp1["boxes"]:  # FWHM_x
        b.set_facecolor(RED_LIGHT)
        b.set_edgecolor("black")
        b.set_linewidth(1.2)

    for b in bp2["boxes"]:  # FWHM_y
        b.set_facecolor(RED_DARK)
        b.set_edgecolor("black")
        b.set_linewidth(1.2)

    for median in bp1["medians"]:
        median.set_color("black")
        median.set_linewidth(1.8)

    for median in bp2["medians"]:
        median.set_color("white")
        median.set_linewidth(1.8)

    ax1.set_xticks(positions_x + 0.35)
    ax1.set_xticklabels(samples, rotation=25, ha="right")
    ax1.set_ylabel("FWHM (nm)")
    ax1.set_title("PSF Boxplots")
    ax1.grid(True, axis="y", alpha=0.25)

    ax1.plot([], [], color=RED_LIGHT, linewidth=10, label="FWHM$_x$")
    ax1.plot([], [], color=RED_DARK, linewidth=10, label="FWHM$_y$")
    ax1.legend(frameon=True, loc="upper left", bbox_to_anchor=(1.02, 1.0))

    safe_label = exposure_label.replace(" ", "")
    out_png = plot_dir / f"PSF_{safe_label}_scatter_box.png"
    out_pdf = plot_dir / f"PSF_{safe_label}_scatter_box.pdf"

    fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)


def make_combined_fwhm_stats_figure(exposure_label: str, data_by_sample: dict[str, pd.DataFrame], plot_dir: Path) -> None:
    """
    Additional figure per exposure:
      - Combined FWHM = (x+y)/2
      - Stats: Mann–Whitney U vs Bead + Holm correction
      - Asterisks (ns hidden)
    """
    for _, df in data_by_sample.items():
        if "fwhm_mean_nm" not in df.columns:
            df["fwhm_mean_nm"] = (df[COL_X] + df[COL_Y]) / 2.0

    samples = list(data_by_sample.keys())
    arrays = [data_by_sample[s]["fwhm_mean_nm"].values for s in samples]

    stats_df = stats_vs_bead(data_by_sample, metric_col="fwhm_mean_nm")
    stats_out = plot_dir / f"PSF_stats_vs_bead_combined_{exposure_label.replace(' ', '')}.csv"
    stats_df.to_csv(stats_out, index=False)

    star_map = {}
    for _, row in stats_df.iterrows():
        s = row["comparison"].split(" vs ")[0]
        star_map[s] = row["stars"]

    plt.rcParams.update(
        {
            "font.family": "Arial",
            "font.size": 12,
            "axes.titlesize": 14,
            "axes.labelsize": 13,
            "xtick.labelsize": 12,
            "ytick.labelsize": 12,
        }
    )

    fig, ax = plt.subplots(figsize=(7.2, 5.8))

    bp = ax.boxplot(
        arrays,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="white", linewidth=2.0),
        boxprops=dict(linewidth=1.2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
    )
    for b in bp["boxes"]:
        b.set_facecolor(RED_DARK)
        b.set_edgecolor("black")

    ax.set_xticks(np.arange(1, len(samples) + 1))
    ax.set_xticklabels(samples, rotation=25, ha="right")
    ax.set_ylabel("Combined FWHM = (FWHM$_x$ + FWHM$_y$)/2 (nm)")
    ax.set_title(f"PSF (Combined FWHM) • {exposure_label}\n(Bead vs treatments; Holm-corrected Mann–Whitney)")
    ax.grid(True, axis="y", alpha=0.25)

    # Stars (hide ns)
    HIDE_NS = True
    y_max = float(np.nanmax(np.concatenate(arrays))) if len(arrays) else 1.0
    y = y_max * 1.04
    step = y_max * 0.05

    for i, sname in enumerate(samples, start=1):
        if sname == "Bead":
            continue
        stars = star_map.get(sname, "ns")
        if HIDE_NS and stars == "ns":
            continue
        ax.text(i, y, stars, ha="center", va="bottom", fontsize=16)
        y += step

    ax.set_ylim(top=y + step)

    safe_label = exposure_label.replace(" ", "")
    out_png = plot_dir / f"PSF_{safe_label}_COMBINED_with_stats.png"
    out_pdf = plot_dir / f"PSF_{safe_label}_COMBINED_with_stats.pdf"
    fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)


def main() -> None:
    base_dir = Path(BASE_DIR)
    if not base_dir.exists():
        raise FileNotFoundError(f"BASE_DIR not found: {base_dir.resolve()}")

    outlier_dir, plot_dir = ensure_dirs(base_dir)

    np.random.seed(RANDOM_SEED)
    cleaned_by_exposure: dict[str, dict[str, pd.DataFrame]] = {k: {} for k in SHEETS.keys()}

    # Read + clean + save CSVs
    for sample_name, fname in FILES.items():
        fpath = base_dir / fname
        if not fpath.exists():
            raise FileNotFoundError(f"File not found: {fpath}")

        for exposure_label, sheet_idx in SHEETS.items():
            df_raw = read_sheet(fpath, sheet_idx)
            df_clean, df_out = split_outliers_iqr_2d(df_raw, k=IQR_K)

            clean_csv = outlier_dir / f"{sample_name}_{exposure_label.replace(' ', '')}_CLEAN.csv"
            out_csv = outlier_dir / f"{sample_name}_{exposure_label.replace(' ', '')}_OUTLIERS.csv"
            df_clean.to_csv(clean_csv, index=False)
            df_out.to_csv(out_csv, index=False)

            cleaned_by_exposure[exposure_label][sample_name] = df_clean

    # Main figures (scatter + x/y boxplots)
    for exposure_label in SHEETS.keys():
        make_exposure_figure(exposure_label, cleaned_by_exposure[exposure_label], plot_dir)

    # Combined FWHM stats figures
    for exposure_label in SHEETS.keys():
        make_combined_fwhm_stats_figure(exposure_label, cleaned_by_exposure[exposure_label], plot_dir)

    # Summary stats CSV
    rows = []
    for exposure_label in SHEETS.keys():
        for sample_name, df in cleaned_by_exposure[exposure_label].items():
            rows.append(
                dict(
                    exposure=exposure_label,
                    sample=sample_name,
                    n_clean=len(df),
                    mean_x_nm=float(df[COL_X].mean()),
                    sd_x_nm=float(df[COL_X].std(ddof=1)),
                    mean_y_nm=float(df[COL_Y].mean()),
                    sd_y_nm=float(df[COL_Y].std(ddof=1)),
                )
            )

    summary = pd.DataFrame(rows)
    summary_path = base_dir / "psf_clean_summary.csv"
    summary.to_csv(summary_path, index=False)

    print("\nDone.")
    print(f"Outliers + cleaned data saved in: {outlier_dir.resolve()}")
    print(f"Plots saved in: {plot_dir.resolve()}")
    print(f"Summary saved as: {summary_path.resolve()}")


if __name__ == "__main__":
    main()
